# 第 4 章：Embedding 与神经语言模型

这个 notebook 对应 `lessons/04_embedding_and_neural_lm.md`，演示 token id 如何查表成向量、哪些 embedding 行收到梯度、pad 行如何冻结，以及 tiny corpus 上的 loss 下降。

In [ ]:
import torch

from src.models.neural_lm import (
    NeuralLanguageModel,
    NeuralLMConfig,
    cosine_similarity_matrix,
    embedding_rows_with_grad,
    train_neural_language_model,
)

## 1. Embedding Shape

`nn.Embedding(vocab_size, hidden_dim)` 把 `LongTensor[B, T]` 查表成 `FloatTensor[B, T, D]`。

In [ ]:
model = NeuralLanguageModel(vocab_size=8, hidden_dim=6, padding_idx=0)
input_ids = torch.tensor([[1, 2, 3], [0, 2, 4]], dtype=torch.long)
hidden = model.embed(input_ids)
logits, _ = model(input_ids)

print("hidden:", hidden.shape)
print("logits:", logits.shape)

## 2. 哪些 Embedding 行收到梯度

Embedding 的梯度只会出现在本 batch 查到过的 token 行上。

In [ ]:
labels = torch.tensor([[2, 3, 4], [1, 3, 5]], dtype=torch.long)
_, loss = model(input_ids, labels)
loss.backward()

print("loss:", round(loss.item(), 4))
print("embedding rows with grad:", sorted(embedding_rows_with_grad(model)))

## 3. Padding Row 不应被更新

设置 `padding_idx=0` 后，pad token 的 embedding 行会保持固定。

In [ ]:
model = NeuralLanguageModel(vocab_size=8, hidden_dim=6, padding_idx=0)
before = model.token_embedding.weight.detach().clone()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
logits, loss = model(input_ids, labels)
optimizer.zero_grad()
loss.backward()
optimizer.step()
after = model.token_embedding.weight.detach()

print("pad row changed:", not torch.allclose(before[0], after[0]))
print("token 1 row changed:", not torch.allclose(before[1], after[1]))

## 4. Tiny Corpus Overfit

小语料上的 loss 下降证明 `embedding -> mixer -> lm_head` 的训练闭环是通的。

In [ ]:
pattern = torch.tensor([1, 2, 3, 4], dtype=torch.long)
token_ids = pattern.repeat(64)
config = NeuralLMConfig(seed=0, hidden_dim=16, batch_size=16, block_size=6, lr=0.03, steps=80)
trained_model, history = train_neural_language_model(token_ids, vocab_size=5, config=config)

print("first loss:", round(history.losses[0], 4))
print("last loss:", round(history.losses[-1], 4))

## 5. Embedding 相似度

相似度矩阵不是语义证明，但它能帮助我们观察 embedding 参数确实变成了可比较的向量。

In [ ]:
similarity = cosine_similarity_matrix(trained_model.token_embedding.weight, [1, 2, 3, 4])
print(similarity.round(decimals=3))